In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
from torch.utils.data import DataLoader

In [3]:
from torch.utils.data import Dataset
from PIL import Image
from pathlib import Path

class CelebALandmarksDataset(Dataset):
    def __init__(self, landmarks_path, img_dir, transform=None):
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.samples = []
        with open(landmarks_path) as f:
            lines = f.readlines()
        # Skip line 0 (count), line 1 (header)
        for line in lines[2:]:
            parts = line.split()
            if len(parts) >= 11:
                fname = parts[0]
                coords = [float(x) for x in parts[1:11]]
                self.samples.append((fname, coords))

        # Keep only samples where the image file exists
        self.samples = [(f, c) for f, c in self.samples if (self.img_dir / f).exists()]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, coords = self.samples[idx]
        img = Image.open(self.img_dir / fname).convert("RGB")
        landmarks = torch.tensor(coords, dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, landmarks

# Paths
landmarks_path = "/home/toru2/Amara/Deep_learning/dl_lab345.ipynb/dataset/landmarks/list_landmarks_align_celeba.txt"
img_dir = "/home/toru2/Amara/Deep_learning/dl_lab345.ipynb/dataset/img_align_celeba"

transform = torchvision.transforms.Compose([
    torchvision.transforms.PILToTensor(),
])
full_dataset = CelebALandmarksDataset(landmarks_path, img_dir, transform=transform)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, test_set = torch.utils.data.random_split(
    full_dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)
print(len(train_set), len(test_set))

162079 40520


In [15]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_chanels, **kwargs):
        super(ConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_chanels, **kwargs)
        self.bn = nn.BatchNorm2d(out_chanels)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))

class InceptionBlock(nn.Module):
    def __init__(self,  in_channels,  out_1x1, red_3x3, out_3x3, red_5x5, out_5x5, out_pool):

        super(InceptionBlock, self).__init__()
        self.branch1 = ConvBlock(in_channels, out_1x1, kernel_size=1)
        self.branch2 = nn.Sequential(
            ConvBlock(in_channels, red_3x3, kernel_size=1, padding=0),
            ConvBlock(red_3x3, out_3x3, kernel_size=3, padding=1))
        self.branch3 = nn.Sequential(
            ConvBlock(in_channels, red_5x5, kernel_size=1),
            ConvBlock(red_5x5, out_5x5, kernel_size=5, padding=2))
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, padding=1, stride=1),
            ConvBlock(in_channels, out_pool, kernel_size=1))

    def forward(self, x):
        branches = (self.branch1, self.branch2, self.branch3, self.branch4)
        return torch.cat([branch(x) for branch in branches], 1)


In [16]:
class InceptionModel(nn.Module):
    def __init__(self, aux = False, residual = True, num_classes = 1000):
        super(InceptionModel, self).__init__()
        self.aux = aux
        self.residual  = residual
        self.dropout = nn.Dropout(p=0.4)
        self.fc = nn.Linear(512, num_classes)

        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.avgpool = nn.AvgPool2d(kernel_size=(7,6), stride=1)

        self.conv1 = ConvBlock(in_channels=3, out_chanels=64, kernel_size=(7,7), stride=(2,2), padding=(3,3))
        self.conv2 = ConvBlock(in_channels=64, out_chanels=192, kernel_size=3, stride=1, padding=1)

        self.incept3a = InceptionBlock(in_channels=192, out_1x1=64, red_3x3=96, out_3x3=128, red_5x5=16, out_5x5=32, out_pool=32)
        self.incept3b = InceptionBlock(in_channels=256, out_1x1=128, red_3x3=128, out_3x3=192, red_5x5=32, out_5x5=112, out_pool=80)

        self.incept4a = InceptionBlock(in_channels=512, out_1x1=192, red_3x3=96, out_3x3=208, red_5x5=16, out_5x5=48, out_pool=64)
        self.incept4b = InceptionBlock(in_channels=512, out_1x1=160, red_3x3=112, out_3x3=224, red_5x5=24, out_5x5=64, out_pool=64)

        self.incept5a = InceptionBlock(in_channels=512, out_1x1=256, red_3x3=160, out_3x3=320, red_5x5=32, out_5x5=128, out_pool=128)
        self.incept5b = InceptionBlock(in_channels=832, out_1x1=128, red_3x3=112, out_3x3=256, red_5x5=32, out_5x5=64, out_pool=64)

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.maxpool(x)
        x = self.incept3a(x)
        x = self.incept3b(x)
        residual = self.maxpool(x)

        x = self.incept4a(residual)
        x = self.incept4b(x)
        x += residual
        residual = self.maxpool(x)

        x = self.incept5a(residual)
        x = self.incept5b(x)
        x += residual
        x = self.avgpool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc(x)

        return x


In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

loader = DataLoader(train_set, batch_size=16, shuffle=False, num_workers=0)
images, landmarks = next(iter(loader))

# Images are already in [0,1] from ToTensor()
images = images.float().to(device)

# Normalize landmarks (178x218)
scale = torch.tensor([178.0, 218.0] * 5, device=device).unsqueeze(0)
targets = (landmarks.to(device) / scale).clamp(0, 1)

model = InceptionModel(num_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

criterion = nn.MSELoss()

model.train()
for step in range(1, 5001):
    optimizer.zero_grad()
    pred = model(images)
    if pred.shape != targets.shape:
        raise RuntimeError(f"Shape mismatch: pred={pred.shape}, targets={targets.shape}")
    loss = criterion(pred, targets)
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(f"step {step}  loss {loss.item():.6f}")
    if loss.item() < 1e-4:
        print("Near-zero loss reached.")
        break

step 100  loss 0.145463
step 200  loss 0.073145
step 300  loss 0.049793
step 400  loss 0.047377
step 500  loss 0.017658
step 600  loss 0.017955
step 700  loss 0.012141
step 800  loss 0.010488
step 900  loss 0.006772
step 1000  loss 0.010167
step 1100  loss 0.007023
step 1200  loss 0.004654
step 1300  loss 0.005438
step 1400  loss 0.003387
step 1500  loss 0.004210
step 1600  loss 0.003798
step 1700  loss 0.002970
step 1800  loss 0.002108
step 1900  loss 0.002400
step 2000  loss 0.001749
step 2100  loss 0.001670
step 2200  loss 0.001410
step 2300  loss 0.001409
step 2400  loss 0.001122
step 2500  loss 0.001338
step 2600  loss 0.001703
step 2700  loss 0.000864
step 2800  loss 0.001098
step 2900  loss 0.000574
step 3000  loss 0.000939
step 3100  loss 0.000727
step 3200  loss 0.000722
step 3300  loss 0.000648
step 3400  loss 0.000503
step 3500  loss 0.000595
step 3600  loss 0.000498
step 3700  loss 0.012707
step 3800  loss 0.006795
step 3900  loss 0.002626
step 4000  loss 0.001713
step 4100